<div dir="rtl">

# ۲ – ساخت RAG ساده روی اخبار فارسی

در این نوت‌بوک، زیرمجموعه‌ی داده‌ی آماده‌شده را به چند بخش کوچک (چانک) تبدیل می‌کنید، برای هر چانک بردارهای embedding می‌سازید، یک ایندکس برداری (مثلاً با FAISS) ایجاد می‌کنید و در نهایت یک سیستم RAG ساده برای پاسخ‌گویی به سؤال‌های فارسی پیاده‌سازی می‌کنید.

</div>


In [2]:
# TODO: import های لازم را بنویسید
# مثال:
import pandas as pd
from pathlib import Path
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
import faiss


OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "c:\Users\Inspiron 5584\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

In [ ]:
# TODO: فایل '../data/processed/news_subset.csv' را بخوانید و چند سطر اول را نمایش دهید
subset_path = Path('../data/processed/news_subset.csv')
df = pd.read_csv(raw_path)

print(df.shape)
df.head()

In [ ]:
# TODO: یک تابع chunking ساده بنویسید که متن (Title + Description) را به قطعه‌های مثلاً 800 کاراکتری با overlap 150 تبدیل کند
# خروجی را در یک DataFrame جدید با ستون‌های: chunk_id, doc_id, text, category, date ذخیره کنید
def chunk_text(
    text: str,
    chunk_size: int = 800,
    overlap: int = 150
): 
    chunks = []
    start = 0
    text_length = len(text)

    while start < text_length:
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start = end - overlap

        if start < 0:
            start = 0

    return chunks

chunk_rows = []

for doc_id, row in df.iterrows():
    text = row["text"]
    category = row["service"]
    date = row["published_datetime"]

    chunks = chunk_text(text)

    for i, chunk in enumerate(chunks):
        chunk_rows.append({
            "chunk_id": f"{doc_id}_{i}",
            "doc_id": doc_id,
            "text": chunk,
            "category": category,
            "date": date
        })
        
df_chunks = pd.DataFrame(chunk_rows)

df_chunks.head()


print("Total documents:", len(df))
print("Total chunks:", len(df_chunks))

In [ ]:
# TODO: دو مدل semantic embedding و lexical embedding را لود کنید و برای هر chunk دو بردار embedding بسازید
# با استفاده از یک روش دلخواه دو embedding را با یکدیکر ترکیب کنید و بردار سوم را بسازید
# همه‌ی embedding ها را در یک آرایه numpy قرار دهید
semantic_model = SentenceTransformer(
    "paraphrase-multilingual-MiniLM-L12-v2"
)

tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    analyzer="word"
)

texts = df_chunks["text"].tolist()

semantic_embeddings = semantic_model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

semantic_embeddings = np.array(semantic_embeddings)

tfidf_embeddings = tfidf_vectorizer.fit_transform(texts)
tfidf_embeddings = tfidf_embeddings.toarray()

alpha = 0.7   
beta = 0.3    

semantic_scaled = semantic_embeddings * alpha
tfidf_scaled = tfidf_embeddings * beta

hybrid_embeddings = np.concatenate(
    [semantic_scaled, tfidf_scaled],
    axis=1
)

print("Semantic shape:", semantic_embeddings.shape)
print("Lexical shape:", tfidf_embeddings.shape)
print("Hybrid shape:", hybrid_embeddings.shape)

In [ ]:
# TODO: با توجه به بعد embedding، یک FAISS IndexFlatIP بسازید و embedding ها را در آن اضافه کنید
embeddings = hybrid_embeddings

faiss.normalize_L2(embeddings)

dim = embeddings.shape[1]
print("Embedding dimension:", dim)

index = faiss.IndexFlatIP(dim)
index.add(embeddings)

print("Total vectors in index:", index.ntotal)
print(len(df_chunks))

In [ ]:
# TODO: تابعی به نام retrieve(question, top_k=5, embedding_model) بنویسید که:
#   1. سوال را embed کند
#   2. از FAISS نزدیک‌ترین chunk ها را پیدا کند
#   3. متن chunk های برتر را برگرداند
def retrieve(question: str, 
             top_k: int = 5, 
             embedding_model=None,   
             tfidf_vectorizer=None,  
             alpha=0.7,
             beta=0.3,
             index=None,
             df_chunks=None):
     
    if embedding_model is None:
        raise ValueError("embedding_model باید مشخص شود")
     
    query_sem = embedding_model.encode([question], normalize_embeddings=True)
    
     
    if tfidf_vectorizer is not None:
        query_tfidf = tfidf_vectorizer.transform([question]).toarray()
        query_hybrid = np.concatenate([query_sem * alpha, query_tfidf * beta], axis=1)
        faiss.normalize_L2(query_hybrid)
        query_vector = query_hybrid.astype('float32')
    else:
        query_vector = query_sem.astype('float32')
     
    D, I = index.search(query_vector, top_k)
    
    
    results = []
    for idx, score in zip(I[0], D[0]):
        row = df_chunks.iloc[idx]
        results.append({
            "chunk_id": row["chunk_id"],
            "text": row["text"],
            "category": row["category"],
            "date": row["date"],
            "score": float(score)
        })
    
    return results

In [ ]:
# TODO: تابع answer(question) بنویسید که از retrieve استفاده کند
# فعلاً می‌توانید پاسخ را فقط با چسباندن متن chunk ها بسازید
def answer(question: str, 
           top_k: int = 5,
           embedding_model=None,
           tfidf_vectorizer=None,
           alpha=0.7,
           beta=0.3,
           index=None,
           df_chunks=None):
     
    top_chunks = retrieve(
        question=question,
        top_k=top_k,
        embedding_model=embedding_model,
        tfidf_vectorizer=tfidf_vectorizer,
        alpha=alpha,
        beta=beta,
        index=index,
        df_chunks=df_chunks
    )
     
    answer_text = "\n\n".join([c["text"] for c in top_chunks])
    
    return answer_text

In [ ]:
# TODO: چند سوال نمونه از خودتان بپرسید و ببینید خروجی معقول است یا خیر
sample_questions = [
    "پیروزی تیم فوتبال تبریز در لیگ برتر چگونه بود؟",
    "جدیدترین اخبار اقتصادی ایران چیست؟",
    "چه رویداد سیاسی مهمی اخیراً رخ داده است؟",
    "اخبار ورزشی امروز شامل چه تیم‌هایی است؟",
    "آخرین خبر درباره بازار ارز و سکه چیست؟"
]

for q in sample_questions:
    print(f"\n--- Question: {q} ---\n")
    ans = answer(
        question=q,
        top_k=3,
        embedding_model=semantic_model,
        tfidf_vectorizer=tfidf_vectorizer,
        alpha=0.7,
        beta=0.3,
        index=index,
        df_chunks=df_chunks
    )
    print(ans[:1000], "...")